In [ ]:
!pip install mne
import torch 
import kagglehub
import pandas as pd
import numpy as np
import mne  
import os
import glob
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
print("Verificando dataset...")
path = kagglehub.dataset_download("sigfest/database-for-emotion-recognition-system-gameemo")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinando usando: {device}")


In [ ]:
pasta_destino = "./data/processed/mne_raw"
os.makedirs(pasta_destino, exist_ok=True)
padrao_busca = os.path.join(path,"GAMEEMO","*","Raw EEG Data",".csv format","*.csv")
todos_arquivos = glob.glob(padrao_busca,recursive=True)
canais_oficiais = ['AF3', 'AF4', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8']

for arquivo in todos_arquivos:
    nome_original = os.path.basename(arquivo)
    testado = nome_original[:3] 
    jogo = nome_original[3:5]

    print(f"Processando: Sujeito {testado} | Jogo {jogo}")

    df = pd.read_csv(arquivo)
    df_limpo = df[canais_oficiais]
    nomes_canais = list(df_limpo.columns)
    df_volts = df_limpo.T*1e-6
    taxa_amostragem=128

    info = mne.create_info(ch_names=nomes_canais,sfreq=taxa_amostragem,ch_types='eeg')
    info.set_montage('standard_1020')
    raw = mne.io.RawArray(df_volts,info)

    nome_arquivo_saida = f"{testado}_{jogo}_raw.fif" 
    caminho_salvamento = os.path.join(pasta_destino, nome_arquivo_saida)
    raw.save(caminho_salvamento, overwrite=True)

    print(f"✅ Salvo com sucesso em: {caminho_salvamento}\n")
    print("\nObjeto MNE criado com sucesso!")
raw.plot_sensors(show_names=True)



In [ ]:
for arquivo in os.listdir(pasta_destino):
    print("1. Carregando o sinal bruto na RAM...")
    caminho_arquivo = os.path.join(pasta_destino,arquivo)
    raw = mne.io.read_raw_fif(caminho_arquivo, preload=True, verbose=False)

    print("2. Aplicando Filtro Passa-Banda (0.5 Hz - 45.0 Hz)...")
    raw.filter(l_freq=0.5,h_freq=45,fir_design='firwin',verbose=False)

    print("3. Cortando em Janelas de 1 segundo (com 50% de sobreposição)...")
    epochs = mne.make_fixed_length_epochs(raw,duration=1.0,overlap=0.5,preload=True,verbose=False)
    matriz3d_numpy = epochs.get_data()

    pasta_numpy = "./data/processed/numpy"
    os.makedirs(pasta_numpy,exist_ok=True)
    nome_numpy = os.path.basename(arquivo)
    testado,jogo,_ = nome_numpy.split("_")
    nome_arquivo = f"{testado}_{jogo}_tensor.npy"
    caminho_numpy = os.path.join(pasta_numpy,nome_arquivo)
    np.save(caminho_numpy,matriz3d_numpy)

    print(f"✅ Tensor 3D salvo com sucesso em: {caminho_numpy}")

In [ ]:
print("1. Iniciando Extração de Features (Feature Engineering)...")

X_list =[]
Y_list =[]
groups_list=[]
for arquivo in os.listdir(pasta_numpy):
    nome = os.path.basename(arquivo)
    caminho_arquivo = os.path.join(pasta_numpy,arquivo)
    sujeito,jogo,_ = nome.split("_")

    array = np.load(caminho_arquivo)
    media = np.mean(array,axis=2)
    desvio = np.std(array,axis=2)
    maximo = np.max(array,axis=2)
    minimo = np.min(array,axis=2)
    features = np.concatenate([media,desvio,maximo,minimo],axis=1)

    X_list.append(features)
    num_lotes = features.shape[0]
    Y_list.extend([jogo] * num_lotes)
    groups_list.extend([sujeito]*num_lotes)

X = np.vstack(X_list)
Y = np.array(Y_list)
groups = np.array(groups_list)

print(f"✅ Matriz 2D final criada! {X.shape[0]} amostras com {X.shape[1]} features.")
print("-" * 40)
print("2. Iniciando Treinamento e Validação Leave-One-Subject-Out (LOSO)...")

logo = LeaveOneGroupOut()
rf = RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1)
acuracias_por_sujeito = []
sujeito_atual = 1

for train_index, test_index in logo.split(X,Y,groups):
    X_test, Y_test = X[test_index], Y[test_index]
    X_train, Y_train = X[train_index],Y[train_index]

    sujeito_teste = groups[test_index][0]

    rf.fit(X_train,Y_train)
    chutado = rf.predict(X_test)
    acuracia = accuracy_score(Y_test,chutado)
    acuracias_por_sujeito.append(acuracia)

    print(f"Rodada {sujeito_atual:02d} | Testando no {sujeito_teste} | Acurácia: {acuracia:.2%}")
    sujeito_atual += 1
print(f"🎯 ACURÁCIA MÉDIA DO BASELINE: {np.mean(acuracias_por_sujeito):.2%}")